In [ ]:
!pip install laspy rasterio matplotlib numpy

In [ ]:
import laspy
import numpy as np

def load_vegetation_points(laz_file: str):
    """
    Loads only vegetation points from a normalized .laz file.


    Caution: As comments in Line 39-46 shows, for difference LiDAR datasets, the vegetation
             points should be correctly specified.
             
    Parameters
    ----------
    laz_file : str
        Path to the normalized LiDAR .laz file.

    Returns
    -------
    x, y, z : np.ndarray
        Arrays of vegetation point coordinates (X, Y) and normalized height (Z).
    """
    try:
        with laspy.open(laz_file) as las_file:
            las = las_file.read()

        # Filter for vegetation points (classification: 3 = low, 4 = medium, 5 = high vegetation)
        #vegetation_mask = np.isin(las.classification, [3, 4, 5])

        # For Dutch AHN case, should be used as below:
        vegetation_mask = np.isin(las.classification, [1])
        
        # Extract only vegetation points
        x = np.array(las.x[vegetation_mask])
        y = np.array(las.y[vegetation_mask])
        z = np.array(las.z[vegetation_mask])  # Normalized height

        print(f"Loaded {len(x)} vegetation points from {laz_file}")
        return x, y, z

    except Exception as e:
        raise Exception(f"Error loading vegetation points: {e}")



In [ ]:
import rasterio
from rasterio.transform import from_origin

def compute_mean_height(x, y, z, resolution=10.0):
    """
    Computes mean vegetation height in a grid of given resolution.

    Parameters
    ----------
    x, y, z : np.ndarray
        Arrays of vegetation point coordinates (X, Y) and normalized height (Z).
    resolution : float, optional
        Grid cell size in meters (default is 10m).

    Returns
    -------
    mean_height_grid : np.ndarray
        The mean vegetation height grid.
    transform : rasterio.transform.Affine
        Transform information for GeoTIFF output.
    """
    # Define grid extent
    min_x, max_x = np.min(x), np.max(x)
    min_y, max_y = np.min(y), np.max(y)

    # Define grid size
    x_bins = np.arange(min_x, max_x + resolution, resolution)
    y_bins = np.arange(min_y, max_y + resolution, resolution)

    # Create empty grid
    mean_height_grid = np.full((len(y_bins)-1, len(x_bins)-1), np.nan)

    # Assign vegetation points to grid cells
    x_indices = np.digitize(x, x_bins) - 1
    y_indices = np.digitize(y, y_bins) - 1

    # Compute mean height per grid cell
    for i in range(len(x)):
        if 0 <= x_indices[i] < mean_height_grid.shape[1] and 0 <= y_indices[i] < mean_height_grid.shape[0]:
            if np.isnan(mean_height_grid[y_indices[i], x_indices[i]]):
                mean_height_grid[y_indices[i], x_indices[i]] = z[i]
            else:
                mean_height_grid[y_indices[i], x_indices[i]] = np.nanmean([mean_height_grid[y_indices[i], x_indices[i]], z[i]])

    # Define raster transformation
    transform = from_origin(min_x, max_y, resolution, resolution)

    return mean_height_grid, transform


In [ ]:
def save_metric_as_geotiff(metric_grid, transform, output_file, crs="EPSG:28992"):
    """
    Saves a vegetation metric grid as a GeoTIFF file.

    Parameters
    ----------
    metric_grid : np.ndarray
        The computed vegetation metric grid.
    transform : rasterio.transform.Affine
        Transform for geospatial referencing.
    output_file : str
        Output file path for the GeoTIFF.
    crs : str, optional
        Coordinate Reference System (default is EPSG:28992 for RD New).

    Returns
    -------
    None
    """
    try:
        with rasterio.open(
            output_file,
            "w",
            driver="GTiff",
            height=metric_grid.shape[0],
            width=metric_grid.shape[1],
            count=1,
            dtype=metric_grid.dtype,
            crs=crs,
            transform=transform
        ) as dst:
            dst.write(metric_grid, 1)

        print(f"✅ Saved: {output_file}")

    except Exception as e:
        raise Exception(f"Error saving GeoTIFF: {e}")


In [ ]:
import rasterio
import matplotlib.pyplot as plt
import numpy as np

def plot_vegetation_metric(tif_file: str, title: str = "Vegetation Metric"):
    """
    Plots a vegetation metric from a GeoTIFF file.

    Parameters
    ----------
    tif_file : str
        Path to the vegetation metric GeoTIFF file.
    title : str, optional
        Title for the plot (default is "Vegetation Metric").

    Returns
    -------
    None
        Displays the vegetation metric plot.
    """
    try:
        # Load the GeoTIFF file
        with rasterio.open(tif_file) as dataset:
            metric_data = dataset.read(1)  # Read first band
            extent = [dataset.bounds.left, dataset.bounds.right, dataset.bounds.bottom, dataset.bounds.top]

        # Handle missing values (replace NaNs for better visualization)
        metric_data = np.where(np.isnan(metric_data), np.nanmin(metric_data), metric_data)

        # Plot the vegetation metric
        plt.figure(figsize=(10, 7))
        plt.imshow(metric_data, extent=extent, origin="upper", cmap="jet")
        plt.colorbar(label="Metric Value")
        plt.title(title)
        plt.xlabel("X Coordinate")
        plt.ylabel("Y Coordinate")
        plt.show()

    except Exception as e:
        raise Exception(f"Error plotting vegetation metric: {e}")


In [ ]:
# File paths
normalized_laz = "../../Datasets/normalized_24HN1_25.laz"
output_tif = "../../Datasets/mean_height.tif"

# Load vegetation points
x, y, z = load_vegetation_points(normalized_laz)

# Compute mean vegetation height in 10m resolution
mean_height_grid, transform = compute_mean_height(x, y, z, resolution=10.0)

# Save as GeoTIFF
save_metric_as_geotiff(mean_height_grid, transform, output_tif)


In [ ]:
plot_vegetation_metric(output_tif, title="Mean vegetation height (10m)")